[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/03_hierarchical/Hierarchical_Clustering.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/03_hierarchical/Hierarchical_Clustering.ipynb)

# Notebook 03 — Hierarchical Clustering
## Building a Tree of Groups

**Dataset**: Iris flowers
**Question**: *How do the flowers naturally organise into a hierarchy of groups?*
**Type**: Unsupervised — agglomerative (bottom-up) clustering

---

## Section 1 — What Is Hierarchical Clustering?

### In plain English

Imagine sorting 150 flower samples onto a table.
You start by saying each flower is its own group (150 tiny groups).
Then you repeatedly merge the two groups that are most similar to each other:

```
Step 1: flower_1 and flower_2 are closest → merge into group_A  (149 groups)
Step 2: flower_5 and flower_7 are next closest → merge into group_B  (148 groups)
...
Step 149: merge the last two remaining groups → 1 group containing all flowers
```

The full history of merges is called a **dendrogram** — a tree diagram.
You can cut the tree at any height to get any number of clusters you want.

### Key advantages

| | K-Means | DBSCAN | **Hierarchical** |
|---|---|---|---|
| Must specify K upfront | Yes | No | **No — choose K by cutting the tree** |
| Shows merge history | No | No | **Yes — dendrogram** |
| Deterministic | No (random init) | Yes | **Yes** |
| Scales to large data | Yes | Yes | Slow for > 10,000 points |

## Section 2 — How Does It Learn?

### Linkage: how to measure distance between groups

When two clusters merge, how do we measure their distance for future merges?

| Linkage | Distance between clusters = | Behaviour |
|---|---|---|
| `ward` | Increase in total variance | Forms compact, equal-size clusters |
| `complete` | Maximum distance between any two points | Conservative, avoids chaining |
| `average` | Mean distance between all pairs | Balanced |
| `single` | Minimum distance (nearest pair) | Can cause "chaining" |

**Ward linkage** is the default and usually works best — it minimises variance within each merged cluster.

### Reading the dendrogram

- Y-axis = distance (cost) at which two groups merged
- **Long vertical line** = the two groups that merged were far apart (natural boundary)
- **Short vertical line** = similar groups merged (same natural cluster)
- Cut the tree horizontally at a long vertical line to find the right K

## Section 3 — What Does the Data Need?

### Clustering is distance-based — scaling is essential

All clustering algorithms group points by how close they are to each other.
Distance is measured in feature space:
```
distance = sqrt((x1_a − x1_b)² + (x2_a − x2_b)² + ...)
```

If `sepal_length` ranges 4–8 cm and `petal_width` ranges 0.1–2.5 cm,
a 1 cm difference in sepal_length contributes equally to distance as a 1 cm difference in petal_width.
But the sepal feature has 4× the range — it dominates the distance calculation.

**Fix: StandardScaler** — all features end up with mean=0 and std=1.
Now every feature contributes equally to the distance.

### What clustering does NOT need

- No target labels (that is the whole point — we do not have them)
- No train/test split (there is no prediction to evaluate on held-out data)
- Only preparation: encode to numbers, fill missing values, scale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Load Iris — a classic dataset with 3 natural groups of flowers
df_raw = sns.load_dataset('iris')
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
TRUE_LABELS = df_raw['species'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

X_raw = df_raw[FEATURES].values

# Scale features — clustering algorithms are distance-based; scaling is required
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

print(f"Iris dataset: {X_raw.shape[0]} flowers, {X_raw.shape[1]} features")
print(f"True groups  : {df_raw['species'].value_counts().to_dict()}")
print()
print("We will PRETEND we do not know the species labels.")
print("The clustering algorithm must discover the 3 groups by itself.")
df_raw.head(6)

## Section 4 — Build the Dendrogram and Cut for Clusters

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, ax=ax, truncate_mode='level', p=5,
           color_threshold=6, above_threshold_color='grey')
ax.axhline(6, color='orangered', ls='--', lw=1.5, label='Cut here → 3 clusters')
ax.set_title('Dendrogram (Ward Linkage)\nLong vertical lines = natural cluster boundaries', fontsize=12)
ax.set_xlabel('Flower samples (truncated)', fontsize=11)
ax.set_ylabel('Merge distance (Ward linkage)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()
print()
print('The three long vertical lines near the top show the three natural groups in the data.')

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
pred_labels = agg.fit_predict(X)

sil = silhouette_score(X, pred_labels)
print(f"Silhouette score (3 clusters, Ward): {sil:.4f}")
print("Cluster sizes:", dict(zip(*np.unique(pred_labels, return_counts=True))))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#5C6BC0', '#26A69A', '#FFA726']

for ax, (labels, title) in zip(axes, [
    (pred_labels, f'Hierarchical Clustering (Ward, K=3)\nSilhouette={sil:.3f}'),
    (TRUE_LABELS,  'True Species')
]):
    for k in range(3):
        mask = labels == k
        name = ['setosa','versicolor','virginica'][k] if ax == axes[1] else f'Cluster {k}'
        ax.scatter(X[mask,2], X[mask,3], color=colors[k], s=40, alpha=0.8, label=name)
    ax.set_xlabel('petal_length (scaled)', fontsize=11)
    ax.set_ylabel('petal_width (scaled)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend()

plt.tight_layout()
plt.show()